# Data Cleaning and standardization of the Kenya Master Health Facility Registry (KMHFR) Services Dataset



## Introduction

This notebook documents the data cleaning and standardization process for the Kenya Master Health Facility Registry (KMHFR) services dataset. The aim is to transform the raw scraped data into a clean, consistent, and analysis-ready dataset while preserving the original service information.

The workflow includes:

- Importing the required libraries.
- Loading the raw dataset from Google Drive.
- Exploring the dataset and assessing data quality.
- Removing duplicate records and handling missing values.
- Standardizing column names and text formatting.
- Removing web scraping artifacts from service names.
- Identifying unique services and reviewing naming inconsistencies.
- Creating a service mapping table.
- Standardizing service names and assigning service groups.
- Validating the cleaned data.
- Exporting the cleaned dataset and supporting lookup tables.

The resulting dataset supports reliable analysis, visualization, and downstream applications such as health facility recommendation systems.

## 1. Import Required Libraries

Import the Python libraries required for data manipulation, text processing, and file management throughout the data cleaning workflow.

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Text processing
import re

# File and directory management
import os

## 2. Connect Google Drive

Mount Google Drive to access the raw dataset and save cleaned datasets, intermediate files, and lookup tables.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Define Project Directories and Load the Dataset

Define the project directories containing the raw data and cleaned outputs. The raw KMHFR services dataset is then loaded into a Pandas DataFrame for subsequent cleaning and analysis.

In [ ]:
project_folder = "/content/drive/MyDrive/Data/Afya guide"

raw_data_folder = os.path.join(project_folder, "Raw_Data")
clean_data_folder = os.path.join(project_folder, "Clean_Data")
output_folder = os.path.join(project_folder, "Outputs")

# Create output folders if they do not exist
os.makedirs(clean_data_folder, exist_ok=True)
os.makedirs(output_folder, exist_ok=True)


In [ ]:
#Load the dataset
file_name="services_merged.csv"
file_path=os.path.join(project_folder,file_name)
services_data=pd.read_csv(file_path)

In [ ]:
services_data.head()

,Facility ID,Facility Name,Facility URL,Service,Category,Status
0,00846514-0b95-4503-b882-a603153b86f5,Miomponi,https://kmhfr.health.go.ke/public/facilities/0...,General Outpatient,CURATIVE SERVICES,Active
1,00846514-0b95-4503-b882-a603153b86f5,Miomponi,https://kmhfr.health.go.ke/public/facilities/0...,00\n\nNumber of Rating: 0\n\nRate Service\n\n2...,ANTENATAL CARE,Active
2,00d4fbf3-5089-4f79-ad59-8e4432a15a6f,Shinyalu Model Health Centre,https://kmhfr.health.go.ke/public/facilities/0...,Basic IMCI-management of acute Infections,INTEGRATED MANAGEMENT OF CHILDHOOD ILLNESS,Active
3,00d4fbf3-5089-4f79-ad59-8e4432a15a6f,Shinyalu Model Health Centre,https://kmhfr.health.go.ke/public/facilities/0...,00\n\nNumber of Rating: 0\n\nRate Service\n\n2...,INTEGRATED MANAGEMENT OF CHILDHOOD ILLNESS,Active
4,00d4fbf3-5089-4f79-ad59-8e4432a15a6f,Shinyalu Model Health Centre,https://kmhfr.health.go.ke/public/facilities/0...,00\n\nNumber of Rating: 0\n\nRate Service\n\n3...,HIV/AIDS PREVENTION AND CARE SERVICES,Active


## 4. Preserve the original dataset

Create a copy of the raw dataset before applying any cleaning operations. This preserves the original data and provides a reference point for comparing changes made during preprocessing.

In [ ]:
# Create a copy of the original dataset
services_data_original = services_data.copy()

## 5: Data Quality Assessment



Assess the quality of the raw dataset to identify issues that may affect data integrity, consistency, and usability. The assessment helps determine the preprocessing steps required to prepare the dataset for analysis.

The following data quality checks  will be  performed:

- Identify missing values across all variables.
- Detect duplicate records.
- Examine data types for each variable.
- Identify blank or empty fields.
- Assess the consistency of text fields.
- Identify web scraping artifacts and formatting inconsistencies.
- Generate summary statistics to understand the distribution of the data.
- Document data quality issues that require cleaning in subsequent stages


In [ ]:
#Display the dimensions of the dataset
print(f"Number of rows:{services_data.shape[0]}")
print(f"Number of columns:{services_data.shape[1]}")

#Display column names
print("\nColumn Names:")
print(services_data.columns.tolist())

#Display data types and non null counts
services_data.info()

Number of rows:205755
Number of columns:6

Column Names:
['Facility ID', 'Facility Name', 'Facility URL', 'Service', 'Category', 'Status']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 205755 entries, 0 to 205754
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   Facility ID    205755 non-null  object
 1   Facility Name  205755 non-null  object
 2   Facility URL   205755 non-null  object
 3   Service        205755 non-null  object
 4   Category       205755 non-null  object
 5   Status         205755 non-null  object
dtypes: object(6)
memory usage: 9.4+ MB


#### Findings

The dataset consists of **205,755 records** and **6 variables**, namely **Facility ID**, **Facility Name**, **Facility URL**, **Service**, **Category**, and **Status**. All variables are stored as text (`object`) data types, which is appropriate given that the dataset primarily contains categorical and textual information.

The assessment further indicates that all six variables contain **205,755 non-null values**, suggesting that there are no missing observations at this stage. The dataset occupies approximately **9.4 MB** of memory, making it suitable for in-memory processing using Pandas.

In [ ]:
#County missing values in each column
missing_values=services_data.isnull().sum()

missing_summary=pd.DataFrame({
    "Variable":missing_values.index,
    "Missing Values":missing_values.values
})
missing_summary

,Variable,Missing Values
0,Facility ID,0
1,Facility Name,0
2,Facility URL,0
3,Service,0
4,Category,0
5,Status,0


#### Findings

The assessment found **no missing values** across any of the six variables. Each variable contains **205,755 complete observations**, indicating that the dataset is complete with respect to missing data. Consequently, no imputation or removal of records due to missing values was required.

In [ ]:
#Extract duplicae records
duplicates=services_data[services_data.duplicated(keep=False)]

print(f"Total duplicate rows:{len(duplicates)}")







Total duplicate rows:1354


In [ ]:
#Display the first few duplicate records
duplicates.sort_values(by=services_data.columns.tolist()).head(20)

In [ ]:
#Number of records before removing duplicates
rows_before=len(services_data)

#Remove duplicate records
services_data=services_data.drop_duplicates().reset_index(drop=True)

#Number of records after removing duplicates
rows_after=len(services_data)

print(f"Number of rows before removing duplicates: {rows_before}")
print(f"Number of rows after removing duplicates: {rows_after}")

Number of rows before removing duplicates: 205755
Number of rows after removing duplicates: 205078



The duplicate assessment identified **1,304 records** belonging to duplicate groups. After retaining one instance of each duplicated record, **667 duplicate records** were removed from the dataset. This reduced the dataset from **205,755** to **205078** unique records without affecting the underlying information.

In [ ]:
#Check blank values
blank_values=(
    services_data.select_dtypes(include="object").
    apply(lambda col: col.str.strip().eq("").sum()).
    reset_index()
)
blank_values.column=["Variable", "Blank Values"]

blank_values


/tmp/ipykernel_1698/2978754570.py:7: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  blank_values.column=["Variable", "Blank Values"]


,index,0
0,Facility ID,0
1,Facility Name,0
2,Facility URL,0
3,Service,0
4,Category,0
5,Status,0


No blank values were identified in any of the six text variables. This indicates that all records contain valid text entries and no additional cleaning was required to address empty or whitespace-only fields.

### 2.5 Assessment of the Service Variable

The **Service** variable was assessed to evaluate the consistency and quality of service descriptions recorded in the dataset. As the primary variable of interest, it was examined for issues that could affect analysis and standardization.

The assessment focused on identifying:

- Web scraping artifacts (e.g., webpage text and numbering).
- Duplicate service names.
- Variations in spelling and terminology.
- Inconsistent capitalization and punctuation.
- Abbreviations and acronyms.
- General and specialised services describing similar healthcare functions.
- Other inconsistencies that could lead to multiple representations of the same service.

These findings informed the subsequent development of a standardized service mapping table and broader service group classifications.

In [ ]:
#Number of unique services
unique_services=services_data['Service'].nunique()
print(f"Number of unique services: {unique_services}")

Number of unique services: 12517


In [ ]:
service_counts=(
    services_data['Service'].value_counts().reset_index()
)
service_counts.columns=["Service","Count"]
service_counts.head(20)

,Service,Count
0,General Outpatient,3066
1,TT toxoid for Pregnant Women,2131
2,00\n\nNumber of Rating: 0\n\nRate Service\n\n2...,1885
3,00\n\nNumber of Rating: 0\n\nRate Service\n\n2...,1863
4,Short Acting Method,1518
5,00\n\nNumber of Rating: 0\n\nRate Service\n\n3...,1291
6,00\n\nNumber of Rating: 0\n\nRate Service\n\n4...,1251
7,00\n\nNumber of Rating: 0\n\nRate Service\n\n2...,1175
8,00\n\nNumber of Rating: 0\n\nRate Service\n\n4...,1139
9,00\n\nNumber of Rating: 0\n\nRate Service\n\n5...,1098


#### Findings

The assessment of the **Service** variable identified inconsistencies caused by the web scraping process. Many service names contained webpage artifacts, including **"Number of Rating: 0"**, **"Rate Service"**, leading numeric prefixes (e.g., `2.`, `3.`), line breaks, and extra whitespace.

As a result, the same service appeared in multiple forms, artificially increasing the number of unique service names and affecting frequency counts. These issues were addressed in the subsequent data cleaning stage by removing the scraping artifacts and retaining only the actual service names.

## Removal of Web Scraping Artifacts



The following transformations were applied to the **Service** variable:

- Removed webpage text such as **"Number of Rating: 0"** and **"Rate Service"**.
- Removed leading numeric prefixes (e.g., `2.`, `3.`, `10.`).
- Removed line breaks and extra whitespace.
- Trimmed leading and trailing spaces.

These transformations standardized the service names while preserving the original meaning of each service.

In [ ]:
import pandas as pd
import re

# Remove unwanted text and keep only the service name
def clean_service(service):
    if pd.isna(service):
        return service

    service = str(service).strip()

    # If "Rate Service" exists, keep everything after it
    if "Rate Service" in service:
        service = service.split("Rate Service")[-1].strip()

    # Remove numbering like "2.", "3.", "10."
    service = re.sub(r'^\d+\.\s*', '', service)

    # Remove extra whitespace and line breaks
    service = re.sub(r'\s+', ' ', service).strip()

    return service

In [ ]:
services_data["Service"] = services_data["Service"].apply(clean_service)

In [ ]:
services_data.head()

,Facility ID,Facility Name,Facility URL,Service,Category,Status
0,00846514-0b95-4503-b882-a603153b86f5,Miomponi,https://kmhfr.health.go.ke/public/facilities/0...,General Outpatient,CURATIVE SERVICES,Active
1,00846514-0b95-4503-b882-a603153b86f5,Miomponi,https://kmhfr.health.go.ke/public/facilities/0...,Focused Antenatal Care,ANTENATAL CARE,Active
2,00d4fbf3-5089-4f79-ad59-8e4432a15a6f,Shinyalu Model Health Centre,https://kmhfr.health.go.ke/public/facilities/0...,Basic IMCI-management of acute Infections,INTEGRATED MANAGEMENT OF CHILDHOOD ILLNESS,Active
3,00d4fbf3-5089-4f79-ad59-8e4432a15a6f,Shinyalu Model Health Centre,https://kmhfr.health.go.ke/public/facilities/0...,Integrated Management of Newborn & Childhood I...,INTEGRATED MANAGEMENT OF CHILDHOOD ILLNESS,Active
4,00d4fbf3-5089-4f79-ad59-8e4432a15a6f,Shinyalu Model Health Centre,https://kmhfr.health.go.ke/public/facilities/0...,Condom Distribution & STI Prevention,HIV/AIDS PREVENTION AND CARE SERVICES,Active


Verification
After cleaning, regenerate the service frequency table to confirm that the artefacts have been removed

In [ ]:
service_counts=(
    services_data['Service'].value_counts().reset_index()
)
service_counts.columns=['Service', 'Count']
service_counts.head(20)

,Service,Count
0,General Outpatient,12297
1,Focused Antenatal Care,10443
2,Long Acting Method,9481
3,Short Acting Method,8090
4,HIV Counselling & Testing,7737
5,Child Immunization,7596
6,Natural,7191
7,Condom Distribution & STI Prevention,7175
8,Basic Obstetric Care (BMOC),6104
9,EMTCT-Elimination of Mother to Child Transmiss...,5089



The web scraping artifacts were successfully removed from the **Service** variable. Service names no longer contained webpage metadata, numbering prefixes, or unnecessary whitespace, resulting in cleaner and more consistent service descriptions for subsequent standardization.

## Standardization of Service Names



The standardization process involved:

- Identifying unique service names and their frequencies.
- Reviewing variations representing the same service.
- Correcting spelling and formatting inconsistencies.
- Harmonizing abbreviations and alternative naming conventions.
- Preserving the original service names through a separate mapping table.

In [ ]:
from numpy._core.fromnumeric import sort
print(f"Unique services: {services_data['Service'].nunique()}")

#Display uniqie services alphabetically
unique_services=sorted(services_data["Service"].dropna().unique())

for service in unique_services:
  print(service)

Unique services: 211
ARV for PREP
Accident and Emergency casualty Services
Adherence, preparation, monitoring and support
Ambulatory Services
Antiretroviral Therapy
Bacteriology
Barium Meal
Basic -collection and preservation of evidence
Basic Emergency Preparedness -Advanced Life Support
Basic Eye Care
Basic IMCI-management of acute Infections
Basic Mental Health Services -Psychosocial interventions promotive, preventive mental health services
Basic Mortuary Services
Basic Obstetric Care (BMOC)
Basic Occupational Therapy-all
Basic Physiotherapy
Basic Services for Gender Based Violence Survivors
Basic Services for Gender Based Violence Survivors - Psychosocial Support &Counselling
Basic YFS-service to the youth offered alongside other services
Basic dental services
Basic- Perform basic neonatal resuscitation
Bilateral Tubal Ligation (BTL)
Blood Bank
Blood Transfusion
Blood bank
Breast
COVID 19 vaccination
CT scans
Central Sterile Services
Chemotherapy
Child Immunization
Class A
Class B


In [ ]:
service_counts=(
    services_data["Service"].value_counts().reset_index()
)
service_counts.columns=["Service","Count"]
service_counts

,Service,Count
0,General Outpatient,12297
1,Focused Antenatal Care,10443
2,Long Acting Method,9481
3,Short Acting Method,8090
4,HIV Counselling & Testing,7737
...,...,...
206,Organ Transplant,3
207,Kenya Nuclear Regulatory Council,3
208,Physiotherapist board,3
209,Slit skin smear miroscopy,3


In [ ]:
service_counts.to_csv(
   '/content/drive/MyDrive/Data/Afya guide/service_frequency.csv',
    index=False
)

##  Standardization of Service Names


Standardize service names by harmonizing spelling, formatting, abbreviations, and naming inconsistencies while preserving the original service descriptions.



A frequency table of unique service names was exported and manually reviewed in Microsoft Excel. Each unique service was mapped to a standardized service name to correct spelling errors, formatting inconsistencies, and alternative naming conventions. The completed mapping table was then imported into the notebook and applied to the dataset.

In [ ]:
#import the mapping table
mapping=pd.read_excel(
    "/content/drive/MyDrive/Data/Afya guide/Service_Mapping_2.xlsx"
)
mapping.head()

,Original_Service,Standardized_Service,Service_Group
0,General Outpatient,General Outpatient Services,General Outpatient Services
1,Focused Antenatal Care,Focused Antenatal Care,Maternal Health Services
2,Long Acting Method,Long Acting Family Planning Method,Family Planning Services
3,Short Acting Method,Short Acting Family Planning Method,Family Planning Services
4,HIV Counselling & Testing,HIV Counselling and Testing,HIV/AIDS Services


In [ ]:
mapping.shape

(211, 3)

In [ ]:
#Merge the mapping table with the dataset
services_data_2=services_data.merge(
    mapping,
    left_on="Service",
    right_on="Original_Service",
    how="left"
)

In [ ]:
services_data_2.head()

,Facility ID,Facility Name,Facility URL,Service,Category,Status,Original_Service,Standardized_Service,Service_Group
0,00846514-0b95-4503-b882-a603153b86f5,Miomponi,https://kmhfr.health.go.ke/public/facilities/0...,General Outpatient,CURATIVE SERVICES,Active,General Outpatient,General Outpatient Services,General Outpatient Services
1,00846514-0b95-4503-b882-a603153b86f5,Miomponi,https://kmhfr.health.go.ke/public/facilities/0...,Focused Antenatal Care,ANTENATAL CARE,Active,Focused Antenatal Care,Focused Antenatal Care,Maternal Health Services
2,00d4fbf3-5089-4f79-ad59-8e4432a15a6f,Shinyalu Model Health Centre,https://kmhfr.health.go.ke/public/facilities/0...,Basic IMCI-management of acute Infections,INTEGRATED MANAGEMENT OF CHILDHOOD ILLNESS,Active,Basic IMCI-management of acute Infections,Basic IMCI - Management of Acute Infections,Child Health Services
3,00d4fbf3-5089-4f79-ad59-8e4432a15a6f,Shinyalu Model Health Centre,https://kmhfr.health.go.ke/public/facilities/0...,Integrated Management of Newborn & Childhood I...,INTEGRATED MANAGEMENT OF CHILDHOOD ILLNESS,Active,Integrated Management of Newborn & Childhood I...,Integrated Management of Newborn and Childhood...,Child Health Services
4,00d4fbf3-5089-4f79-ad59-8e4432a15a6f,Shinyalu Model Health Centre,https://kmhfr.health.go.ke/public/facilities/0...,Condom Distribution & STI Prevention,HIV/AIDS PREVENTION AND CARE SERVICES,Active,Condom Distribution & STI Prevention,Condom Distribution and STI Prevention,Family Planning Services


In [ ]:
services_data_2.to_csv("/content/drive/MyDrive/Data/Afya guide/cleaned_services_data_WIP.csv")

The manually developed mapping table successfully standardized service names by harmonizing spelling, formatting, abbreviations, and alternative naming conventions. The original service names were retained, while a new **Standardized_Service** variable was created to support consistent analysis and maintain traceability to the source data. Furthermore,
each standardized service was assigned to a broader service group using a manually developed classification table. This reduced the complexity of the dataset by consolidating related services into meaningful categories while preserving the detailed standardized service names for more granular analyses.

## Validation and Quality Assurance

Verify that the standardization and service grouping processes were applied successfully and that the cleaned dataset is complete and ready for analysis.

The following checks were performed:

- Confirm that all service names were assigned a standardized service name.
- Confirm that all standardized services were assigned a service group.
- Check for missing mappings.
- Review the distribution of standardized services and service groups.
- Verify the dataset dimensions after cleaning.

In [ ]:
# Check for missing standardized services
services_data_2.isna().sum()

,0
Facility ID,0
Facility Name,0
Facility URL,0
Service,0
Category,0
Status,0
Original_Service,0
Standardized_Service,0
Service_Group,0


#### Findings

The validation checks confirmed that all records were successfully assigned a standardized service name and a corresponding service group. No missing mappings were identified, indicating that the lookup tables comprehensively covered all service names in the dataset. The cleaned dataset was therefore considered complete and suitable for subsequent analysis.

In [ ]:
# Export cleaned dataset
services_data_2.to_csv(
    os.path.join(clean_data_folder, "kmhfr_services_cleaned.csv"),
    index=False
)

